# Spam Detection - Data Preprocessing

## 1. Introduction

This notebook performs data preprocessing to prepare the SMS spam dataset for
deep learning models. The preprocessing pipeline includes text cleaning,
duplicate removal, and dataset splitting.

**Purpose**: Transform raw text data into clean, structured data ready for
model training and evaluation.

**Key findings from EDA**:
- 5,572 SMS messages (ham/spam)
- 403 duplicates (7.23%)
- Class imbalance: 86.6% ham, 13.4% spam (ratio 6.46:1)
- Spam characteristics: longer messages, phone numbers, URLs, exclamation
  marks, capitals
- Ham characteristics: short, conversational, personal pronouns

### Import Libraries

In [1]:
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Data

Load the spam dataset and apply the same cleaning steps as in EDA.

In [2]:
# Load dataset with latin-1 encoding
df = pd.read_csv(
    "../data/input/spam.csv",
    encoding="latin-1",
)

print(f"Original dataset shape: {df.shape}")

# Rename columns and drop empty ones
df = df.rename(columns={"v1": "label", "v2": "text"})
df = df[["label", "text"]]

print(f"After cleaning: {df.shape}")
print(f"\nClass distribution:")
print(df["label"].value_counts())
df.head()

Original dataset shape: (5572, 5)
After cleaning: (5572, 2)

Class distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 3. Text Cleaning Pipeline

Create a reusable cleaning function for text preprocessing.

**Deep Learning Consideration**: Unlike traditional ML approaches that rely on
hand-crafted features, deep learning models can learn patterns from raw text.
Therefore, we apply minimal preprocessing:

- **Keep stopwords**: Words like "free", "call" are spam indicators
- **Keep punctuation**: Exclamation marks and capitals signal urgency
- **Preserve URLs/phones**: Can be learned as patterns by the model

However, we still apply basic cleaning for consistency and noise reduction.

In [3]:
def clean_text(text, remove_urls=True, remove_phones=True):
    """Clean text data for deep learning models.

    Applies minimal preprocessing suitable for deep learning, preserving
    information that models can learn from (stopwords, some punctuation).

    Args:
        text: Input text string.
        remove_urls: Whether to replace URLs with token. Default True.
        remove_phones: Whether to replace phone numbers with token.
            Default True.

    Returns:
        Cleaned text string.
    """
    if not isinstance(text, str):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Replace URLs with token (optional)
    if remove_urls:
        url_pattern = re.compile(
            r"http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|"
            r"[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+",
            re.VERBOSE,
        )
        text = url_pattern.sub("URL", text)
        text = re.sub(r"www\.[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", "URL", text)

    # Replace phone numbers with token (optional)
    if remove_phones:
        text = re.sub(
            r"\b\d{5,}\b|\b\d{3}[-.]?\d{3}[-.]?\d{4}\b", "PHONE", text
        )

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


print("Text cleaning function defined.")

Text cleaning function defined.


In [4]:
# Test cleaning function
test_texts = [
    "FREE entry in 2 a wkly comp!! Call 09061701461 NOW!",
    "Hey! How are you? Visit www.example.com for more info",
    "Ok lar... Joking wif u oni...",
]

print("Testing text cleaning function:")
print("=" * 70)
for text in test_texts:
    cleaned = clean_text(text)
    print(f"Original: {text}")
    print(f"Cleaned:  {cleaned}")
    print("-" * 70)

Testing text cleaning function:
Original: FREE entry in 2 a wkly comp!! Call 09061701461 NOW!
Cleaned:  free entry in 2 a wkly comp!! call PHONE now!
----------------------------------------------------------------------
Original: Hey! How are you? Visit www.example.com for more info
Cleaned:  hey! how are you? visit URL for more info
----------------------------------------------------------------------
Original: Ok lar... Joking wif u oni...
Cleaned:  ok lar... joking wif u oni...
----------------------------------------------------------------------


In [5]:
# Apply cleaning to dataset
df["text_cleaned"] = df["text"].apply(clean_text)

print(f"Text cleaning completed.")
print(f"\nSample of cleaned texts:")
df[["text", "text_cleaned"]].head(10)

Text cleaning completed.

Sample of cleaned texts:


,text,text_cleaned
0,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ..."
1,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...
4,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro..."
5,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey there darling it's been 3 week's n...
6,Even my brother is not like to speak with me. ...,even my brother is not like to speak with me. ...
7,As per your request 'Melle Melle (Oru Minnamin...,as per your request 'melle melle (oru minnamin...
8,WINNER!! As a valued network customer you have...,winner!! as a valued network customer you have...
9,Had your mobile 11 months or more? U R entitle...,had your mobile 11 months or more? u r entitle...


## 4. Handle Duplicates

Remove duplicate messages to avoid data leakage between train/val/test sets.
We keep the first occurrence of each duplicate.

In [6]:
# Check duplicates before removal
print(f"Dataset size before duplicate removal: {len(df):,}")
duplicates = df.duplicated(subset=["text_cleaned"])
print(f"Number of duplicates: {duplicates.sum():,}")
print(f"Percentage: {duplicates.sum() / len(df) * 100:.2f}%")

# Remove duplicates (keep first occurrence)
df_clean = df.drop_duplicates(subset=["text_cleaned"], keep="first")

print(f"\nDataset size after duplicate removal: {len(df_clean):,}")
print(f"Removed: {len(df) - len(df_clean):,} duplicates")

# Verify class distribution is maintained
print(f"\nClass distribution after duplicate removal:")
print(df_clean["label"].value_counts())
print(f"\nPercentages:")
print(df_clean["label"].value_counts(normalize=True) * 100)

Dataset size before duplicate removal: 5,572
Number of duplicates: 430
Percentage: 7.72%

Dataset size after duplicate removal: 5,142
Removed: 430 duplicates

Class distribution after duplicate removal:
label
ham     4515
spam     627
Name: count, dtype: int64

Percentages:
label
ham     87.806301
spam    12.193699
Name: proportion, dtype: float64


## 5. Train/Validation/Test Split

Split the dataset into train (70%), validation (15%), and test (15%) sets
using stratified sampling to maintain class distribution across splits.

**Why stratified?** With class imbalance (6.46:1), random splits could result
in different class distributions across sets, affecting model evaluation.

In [7]:
# Prepare data for splitting
X = df_clean["text_cleaned"].values
y = df_clean["label"].values

# Convert labels to binary (0=ham, 1=spam)
y_binary = (y == "spam").astype(int)

print(f"Total samples: {len(X):,}")
print(f"Label distribution: {np.unique(y_binary, return_counts=True)}")

Total samples: 5,142
Label distribution: (array([0, 1]), array([4515,  627]))


In [8]:
# First split: 70% train, 30% temp (for val+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_binary, test_size=0.30, random_state=42, stratify=y_binary
)

# Second split: Split temp into 50% val, 50% test (15% each of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train set size: {len(X_train):,} "
      f"({len(X_train) / len(X) * 100:.1f}%)")
print(f"Validation set size: {len(X_val):,} "
      f"({len(X_val) / len(X) * 100:.1f}%)")
print(f"Test set size: {len(X_test):,} "
      f"({len(X_test) / len(X) * 100:.1f}%)")

Train set size: 3,599 (70.0%)
Validation set size: 771 (15.0%)
Test set size: 772 (15.0%)


In [9]:
# Verify stratification worked correctly
print("=" * 70)
print("CLASS DISTRIBUTION VERIFICATION")
print("=" * 70)

for split_name, y_split in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test),
]:
    ham_count = (y_split == 0).sum()
    spam_count = (y_split == 1).sum()
    total = len(y_split)

    print(f"\n{split_name} Set:")
    print(f"  Ham:  {ham_count:,} ({ham_count / total * 100:.2f}%)")
    print(f"  Spam: {spam_count:,} ({spam_count / total * 100:.2f}%)")
    print(f"  Ratio (ham:spam): {ham_count / spam_count:.2f}:1")

print("\n" + "=" * 70)

CLASS DISTRIBUTION VERIFICATION

Train Set:
  Ham:  3,160 (87.80%)
  Spam: 439 (12.20%)
  Ratio (ham:spam): 7.20:1

Validation Set:
  Ham:  677 (87.81%)
  Spam: 94 (12.19%)
  Ratio (ham:spam): 7.20:1

Test Set:
  Ham:  678 (87.82%)
  Spam: 94 (12.18%)
  Ratio (ham:spam): 7.21:1



## 6. Tokenization Strategy Discussion

### Tokenization Options for Deep Learning

**Option 1: Custom Tokenizer (Simple Models)**
- Build vocabulary from training data
- Convert words to integer indices
- Suitable for: LSTM, GRU, CNN, simple embeddings
- Pros: Full control, lightweight, fast
- Cons: No transfer learning benefits

**Option 2: Pre-trained Tokenizer (Transfer Learning)**
- Use tokenizer from pre-trained models (BERT, DistilBERT, RoBERTa)
- Leverage subword tokenization (WordPiece, BPE)
- Suitable for: Transformer models, fine-tuning
- Pros: Better handling of rare words, transfer learning
- Cons: Larger model size, slower inference

**Decision for This Project**:
We will implement both approaches in separate notebooks:
- Simple models: Custom tokenizer (done per-model)
- Transformer models: Pre-trained tokenizer (done per-model)

This preprocessing notebook prepares the clean text data. Tokenization will
be performed in the modeling notebooks based on the chosen architecture.

## 7. Save Preprocessed Data

Save the train/validation/test splits as CSV files for use in modeling
notebooks.

In [10]:
# Create output directory if it doesn't exist
import os

output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

print(f"Output directory: {output_dir}")
print(f"Directory exists: {os.path.exists(output_dir)}")

Output directory: ../data/processed
Directory exists: True


In [11]:
# Create DataFrames for each split
train_df = pd.DataFrame({"text": X_train, "label": y_train})
val_df = pd.DataFrame({"text": X_val, "label": y_val})
test_df = pd.DataFrame({"text": X_test, "label": y_test})

# Save to CSV
train_df.to_csv(f"{output_dir}/train.csv", index=False)
val_df.to_csv(f"{output_dir}/val.csv", index=False)
test_df.to_csv(f"{output_dir}/test.csv", index=False)

print("Data saved successfully!")
print(f"\nFiles created:")
print(f"  - {output_dir}/train.csv ({len(train_df):,} samples)")
print(f"  - {output_dir}/val.csv ({len(val_df):,} samples)")
print(f"  - {output_dir}/test.csv ({len(test_df):,} samples)")

Data saved successfully!

Files created:
  - ../data/processed/train.csv (3,599 samples)
  - ../data/processed/val.csv (771 samples)
  - ../data/processed/test.csv (772 samples)


In [12]:
# Verify saved files
print("Verifying saved files...\n")

for filename in ["train.csv", "val.csv", "test.csv"]:
    filepath = f"{output_dir}/{filename}"
    df_check = pd.read_csv(filepath)
    print(f"{filename}:")
    print(f"  Shape: {df_check.shape}")
    print(f"  Columns: {df_check.columns.tolist()}")
    print(f"  Sample: {df_check.iloc[0]['text'][:50]}...")
    print()

Verifying saved files...

train.csv:
  Shape: (3599, 2)
  Columns: ['text', 'label']
  Sample: painful words- \i thought being happy was the most...

val.csv:
  Shape: (771, 2)
  Columns: ['text', 'label']
  Sample: dating:i have had two of these. only started after...

test.csv:
  Shape: (772, 2)
  Columns: ['text', 'label']
  Sample: haven't found a way to get another app for your ph...



## 8. Summary Statistics

Final verification of the preprocessed dataset.

In [13]:
print("=" * 70)
print("PREPROCESSING SUMMARY")
print("=" * 70)

print(f"\nOriginal dataset size: {len(df):,}")
print(f"After duplicate removal: {len(df_clean):,}")
print(f"Duplicates removed: {len(df) - len(df_clean):,}")

print(f"\nFinal splits:")
print(f"  Train:      {len(X_train):,} samples (70.0%)")
print(f"  Validation: {len(X_val):,} samples (15.0%)")
print(f"  Test:       {len(X_test):,} samples (15.0%)")
print(f"  Total:      {len(X_train) + len(X_val) + len(X_test):,} samples")

print(f"\nClass distribution (overall):")
ham_pct = (y_binary == 0).sum() / len(y_binary) * 100
spam_pct = (y_binary == 1).sum() / len(y_binary) * 100
print(f"  Ham:  {ham_pct:.2f}%")
print(f"  Spam: {spam_pct:.2f}%")

print("\n" + "=" * 70)

PREPROCESSING SUMMARY

Original dataset size: 5,572
After duplicate removal: 5,142
Duplicates removed: 430

Final splits:
  Train:      3,599 samples (70.0%)
  Validation: 771 samples (15.0%)
  Test:       772 samples (15.0%)
  Total:      5,142 samples

Class distribution (overall):
  Ham:  87.81%
  Spam: 12.19%



In [14]:
# Verify no data leakage (no duplicate texts across splits)
print("Data leakage check...\n")

train_set = set(X_train)
val_set = set(X_val)
test_set = set(X_test)

train_val_overlap = train_set.intersection(val_set)
train_test_overlap = train_set.intersection(test_set)
val_test_overlap = val_set.intersection(test_set)

print(f"Train-Val overlap: {len(train_val_overlap)} texts")
print(f"Train-Test overlap: {len(train_test_overlap)} texts")
print(f"Val-Test overlap: {len(val_test_overlap)} texts")

if (
    len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
):
    print("\n✓ No data leakage detected!")
else:
    print("\n✗ WARNING: Data leakage detected!")

Data leakage check...

Train-Val overlap: 0 texts
Train-Test overlap: 0 texts
Val-Test overlap: 0 texts

✓ No data leakage detected!


In [15]:
print("\n" + "=" * 70)
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 70)
print("\nNext steps:")
print("  1. Build custom tokenizer for simple models (LSTM, CNN)")
print("  2. Fine-tune pre-trained models (BERT, DistilBERT)")
print("  3. Train and evaluate models")
print("  4. Compare performance metrics")
print("=" * 70)


PREPROCESSING COMPLETED SUCCESSFULLY

Next steps:
  1. Build custom tokenizer for simple models (LSTM, CNN)
  2. Fine-tune pre-trained models (BERT, DistilBERT)
  3. Train and evaluate models
  4. Compare performance metrics
